In [ ]:
import json
import requests
from urllib.parse import urlparse
from tqdm import tqdm


INPUT_FILE = r"../Data/raw/metadata_wlasl_msasl_merged.jsonl"
OUTPUT_FILE = r"../Data/Processed/url_availability_report.jsonl"


def classify_url(url):
    if not url:
        return "missing"

    domain = urlparse(url).netloc.lower()

    if "youtube.com" in domain or "youtu.be" in domain:
        return "youtube"

    if "google" in domain:
        return "google"

    return domain


def check_url(url):
    if not url:
        return False, "missing"

    try:
        r = requests.head(
            url,
            timeout=10,
            allow_redirects=True
        )

        if r.status_code < 400:
            return True, r.status_code
        else:
            return False, r.status_code

    except Exception as e:
        return False, str(e)


results = []

with open(INPUT_FILE, "r", encoding="utf-8") as f:

    lines = f.readlines()


for line in tqdm(lines):

    item = json.loads(line)

    url = item.get("url")

    available, status = check_url(url)

    result = {
        "index": item.get("index"),
        "label": item.get("label"),
        "text": item.get("text"),
        "signer_id": item.get("signer_id"),
        "split": item.get("split"),

        "url": url,

        "url_type": classify_url(url),

        "available": available,
        "http_status": status,

        "landmark_available": item.get("landmark_available"),
        "landmark_status": item.get("landmark_status")
    }

    results.append(result)


with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for r in results:
        f.write(json.dumps(r) + "\n")


print("Done")
print("WLASL entries checked:", len(results))

print("\nURL types:")
from collections import Counter

print(Counter(r["url_type"] for r in results))

print("\nAvailability:")
print(Counter(r["available"] for r in results))

In [ ]:
import json
import requests
from tqdm import tqdm
from collections import Counter
from urllib3.exceptions import InsecureRequestWarning

requests.packages.urllib3.disable_warnings(
    category=InsecureRequestWarning
)


INPUT_REPORT = r"../Data/Processed/url_availability_report.jsonl"
OUTPUT_REPORT = r"../Data/Processed/retry_failed_report.jsonl"


def retry_check_url(url):

    if not url:
        return False, "missing"

    headers = {
        "User-Agent": (
            "Mozilla/5.0 "
            "(Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 "
            "(KHTML, like Gecko) "
            "Chrome/120 Safari/537.36"
        )
    }

    try:
        r = requests.get(
            url,
            headers=headers,
            timeout=20,
            allow_redirects=True,
            stream=True,
            verify=False
        )

        status = r.status_code

        # We only need to know if the file exists
        if status < 400:
            return True, status

        return False, status

    except Exception as e:
        return False, str(e)



# Load only failed entries

failed_entries = []

with open(INPUT_REPORT, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)

        if item["available"] is False:
            failed_entries.append(item)


print("Failed URLs to retry:", len(failed_entries))


# Retry


retry_results = []

for item in tqdm(failed_entries):

    ok, status = retry_check_url(item["url"])

    new_item = item.copy()

    new_item["retry_available"] = ok
    new_item["retry_status"] = status

    retry_results.append(new_item)



# Save

with open(OUTPUT_REPORT, "w", encoding="utf-8") as f:
    for item in retry_results:
        f.write(json.dumps(item) + "\n")


print("\nDone")
print("Retried:", len(retry_results))


print("\nRetry availability:")
print(
    Counter(
        x["retry_available"]
        for x in retry_results
    )
)


print("\nRetry status:")
print(
    Counter(
        str(x["retry_status"])
        for x in retry_results
    ).most_common(20)
)

In [ ]:
import json
from collections import Counter
from urllib.parse import urlparse

counter = Counter()

with open(r"../Data/Processed/retry_failed_report.jsonl","r",encoding="utf8") as f:
    for line in f:
        x = json.loads(line)

        if x["retry_status"] == 403:
            counter[urlparse(x["url"]).netloc] += 1

print(counter)

In [ ]:
import json

FIRST_REPORT = r"../Data/Processed/url_availability_report.jsonl"
RETRY_REPORT = r"../Data/Processed/retry_failed_report.jsonl"

INPUT_DATASET = r"../Data/raw/metadata_wlasl_msasl_merged.jsonl"
OUTPUT_DATASET = r"../Data/Processed/downloadable.jsonl"


available = {}

with open(FIRST_REPORT, "r", encoding="utf-8") as f:
    for line in f:
        x = json.loads(line)
        available[x["index"]] = x["available"]

# Retry overwrites first pass
with open(RETRY_REPORT, "r", encoding="utf-8") as f:
    for line in f:
        x = json.loads(line)
        available[x["index"]] = x["available"]

print("Downloadable videos:", sum(available.values()))


count = 0

with open(INPUT_DATASET, "r", encoding="utf-8") as fin, \
     open(OUTPUT_DATASET, "w", encoding="utf-8") as fout:

    for line in fin:
        item = json.loads(line)

        if item["dataset_source"] != "wlasl":
            continue

        if available.get(item["index"], False):
            fout.write(json.dumps(item) + "\n")
            count += 1

print(f"Saved {count} entries to {OUTPUT_DATASET}")

In [ ]:
!python extract_landmarks.py --input "./Data/Processed/downloadable.jsonl" --landmarks_dir "./Data/Processed/landmarks" --metadata_out "./Data/Processed/metadata.jsonl" --fail_log_out "./Data/Processed/fail.jsonl" --workers 4 --max_frames 200                                      

In [25]:
import json
from collections import Counter

input_path = "../Data/Processed/metadata.jsonl"
output_path = "../Data/Processed/metadata_filtered.jsonl"

# load
with open(input_path, encoding="utf-8") as f:
    meta = [json.loads(line) for line in f]

print("Original samples:", len(meta))

# count classes
counts = Counter(x["label"] for x in meta)

# choose threshold
MIN_SAMPLES = 3   # this matches your previous split

# keep only classes with enough samples
valid_classes = {
    label for label, c in counts.items()
    if c >= MIN_SAMPLES
}

filtered = [
    x for x in meta
    if x["label"] in valid_classes
]

print("After filtering:")
print("Samples:", len(filtered))
print("Classes:", len(valid_classes))


# save filtered dataset
with open(output_path, "w", encoding="utf-8") as f:
    for x in filtered:
        f.write(json.dumps(x) + "\n")

print("Saved:", output_path)

Original samples: 27353
After filtering:
Samples: 27150
Classes: 1938
Saved: ../Data/Processed/metadata_filtered.jsonl


In [26]:
import json
from collections import Counter

input_file = "../Data/Processed/metadata_filtered.jsonl"
output_file = "../Data/Processed/filtered_remapped.jsonl"


# load
with open(input_file, encoding="utf-8") as f:
    data = [json.loads(line) for line in f]


# check classes
counts = Counter(x["label"] for x in data)

old_labels = sorted(counts.keys())


# create remap
label_map = {
    old:i 
    for i,old in enumerate(old_labels)
}


# apply
for x in data:
    x["label_original"] = x["label"]
    x["label"] = label_map[x["label"]]


# save
with open(output_file,"w",encoding="utf-8") as f:
    for x in data:
        f.write(json.dumps(x)+"\n")


print("Classes:", len(label_map))
print("Min label:", min(x["label"] for x in data))
print("Max label:", max(x["label"] for x in data))

Classes: 1938
Min label: 0
Max label: 1937


In [28]:
!python signer_split_and_augment_plan.py --input "../Data/Processed/filtered_remapped.jsonl" --out_dir "../Data/Processed/split2" --train_frac 0.85 --min_train_signers 3 --min_heldout_signers 2 --mid_low 15 --mid_high 29 --target 30

Loading entries...
  27150 entries loaded

Ranking signers globally...
  203 signers
 Top 5: [('MS-ASL:0', 1662), ('MS-ASL:12', 1553), ('MS-ASL:9', 1530), ('MS-ASL:26', 1439), ('WLASL:5', 1296)]

Pool sizes:
 train: 50
 val: 77
 test: 76

Sample split:
Counter({'train': 23077, 'val': 2065, 'test': 2008})
Written: ..\Data\Processed\split2\split_assignment.jsonl

Checking classes...
Passing classes: 669 / 1938
Written: ..\Data\Processed\split2\class_split_report.json

Creating augmentation plan...
Classes augmented: 334
Synthetic samples: 2882
Written: ..\Data\Processed\split2\augmentation_plan.json


In [29]:
!python create_baseline_dataset.py --split_file "../Data/Processed/split2/split_assignment.jsonl" --report_file "../Data/Processed/split2/class_split_report.json" --out_dir "../Data/Processed/split2/baseline"

Loading split file...
Samples loaded: 27150

Loading class report...
Valid classes: 669
Samples after class filtering: 17613

Final dataset
-----------------------------
Train: 14057
Val: 1801
Test: 1755

Classes
-----------------------------
Train classes: 669
Val classes: 594
Test classes: 596

Signers
-----------------------------
Train signers: 50
Val signers: 75
Test signers: 74

Top 10 classes by train samples
29 need 60
1 nice 56
14 bird 51
28 table 51
3 eat 49
10 school 49
4 no 48
18 sit 48
19 mother 47
7 orange 47

Written:
..\Data\Processed\split2\baseline\train_baseline.jsonl
..\Data\Processed\split2\baseline\val_baseline.jsonl
..\Data\Processed\split2\baseline\test_baseline.jsonl


In [31]:
import json
from collections import defaultdict


files = [
    "train_baseline.jsonl",
    "val_baseline.jsonl",
    "test_baseline.jsonl"
]

base="../Data/Processed/split2/baseline/"


# collect labels from train only
labels=set()

with open(base+"train_baseline.jsonl") as f:
    for line in f:
        x=json.loads(line)
        labels.add(x["label"])


# create compact mapping
labels=sorted(labels)

remap={
    old:new
    for new,old in enumerate(labels)
}


print("New classes:",len(remap))
print(remap)


# rewrite files
for file in files:

    output=base+"remapped_"+file

    with open(base+file) as fin, open(output,"w") as fout:

        for line in fin:

            x=json.loads(line)

            x["label_original"]=x["label"]

            x["label"]=remap[x["label"]]

            fout.write(
                json.dumps(x)+"\n"
            )

print("done")

New classes: 669
{0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7, 8: 8, 9: 9, 10: 10, 11: 11, 12: 12, 13: 13, 14: 14, 15: 15, 16: 16, 17: 17, 18: 18, 19: 19, 20: 20, 21: 21, 22: 22, 23: 23, 24: 24, 25: 25, 26: 26, 27: 27, 28: 28, 29: 29, 30: 30, 31: 31, 32: 32, 33: 33, 34: 34, 35: 35, 36: 36, 37: 37, 38: 38, 39: 39, 40: 40, 41: 41, 42: 42, 43: 43, 44: 44, 45: 45, 46: 46, 47: 47, 48: 48, 49: 49, 50: 50, 51: 51, 52: 52, 53: 53, 54: 54, 55: 55, 56: 56, 57: 57, 58: 58, 59: 59, 60: 60, 61: 61, 62: 62, 63: 63, 64: 64, 65: 65, 66: 66, 67: 67, 68: 68, 69: 69, 70: 70, 71: 71, 72: 72, 73: 73, 74: 74, 75: 75, 76: 76, 77: 77, 78: 78, 79: 79, 80: 80, 81: 81, 82: 82, 83: 83, 84: 84, 85: 85, 86: 86, 87: 87, 88: 88, 89: 89, 90: 90, 91: 91, 92: 92, 93: 93, 94: 94, 95: 95, 96: 96, 97: 97, 98: 98, 99: 99, 100: 100, 101: 101, 102: 102, 103: 103, 104: 104, 105: 105, 106: 106, 107: 107, 108: 108, 109: 109, 110: 110, 111: 111, 112: 112, 113: 113, 114: 114, 115: 115, 116: 116, 117: 117, 118: 118, 119: 119, 12

In [32]:
for split in ["train","val","test"]:

    labels=set()

    with open(
        f"../Data/Processed/remapped_{split}_baseline.jsonl"
    ) as f:

        for line in f:
            labels.add(json.loads(line)["label"])


    print(split)
    print(len(labels))
    print(min(labels),max(labels))

train
665
0 664
val
604
0 664
test
591
0 663
